# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

**Autor:** Jerónimo Zapata Cuadros (`JCuAdrox`) · SI3003, EAFIT

Copia del notebook `03_rag_workshop_papers_nvidia_nim_colab.ipynb` del curso con las tareas **B1–B5**, la reflexión **R1** y los retos **C3** (Gradio con filtro por paper) y **C5** (MMR).

- **Attention Is All You Need** — Vaswani et al. (2017), arXiv 1706.03762
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020), arXiv 2005.11401

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | `Document` y chunking 900/120 |
| 3 | Embeddings NVIDIA e índice Qdrant |
| 4 | Pipeline RAG con LCEL (k=5, umbral 0.25) |
| 5 + Bonus | Quiz con *ground truth* (**B1**) |
| B2–B5 | Hiperparámetros, preguntas trampa, español, quiz propio |
| C3, C5 | Gradio con filtro por paper · MMR frente a similitud |

**Modelos:** generación `nvidia/nemotron-3.5-lightning-30b-a3b` · embeddings `nvidia/llama-nemotron-embed-vl-1b-v2`.

## Parte 0 — Instalar y configurar

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -qU "markitdown[pdf]" gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client requests

In [ ]:
import os
import sys
import time
import json
import getpass
import concurrent.futures
from pathlib import Path
from typing import List, Dict

IN_COLAB = "google.colab" in sys.modules


def find_repo_root() -> Path:
    """Raíz del repo (donde está RESPUESTAS.md); en Colab, el directorio actual."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "RESPUESTAS.md").exists():
            return base
    return Path.cwd()


ROOT = find_repo_root()
EVIDENCIAS = ROOT / "evidencias"
EVIDENCIAS.mkdir(exist_ok=True)

NVIDIA_API_KEY = None
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY and IN_COLAB:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/ y expórtala como variable de entorno.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY  # solo en memoria; nunca se imprime
GENERATION_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
print("✅ Configuración NVIDIA NIM lista | raíz:", ROOT.name)

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA


class ResilientEmbeddings(Embeddings):
    """Envuelve NVIDIAEmbeddings con timeout corto, reintentos y caché.

    El endpoint de embeddings a veces responde 502 tras ~30 s. En lugar de esperar,
    cortamos a los `timeout` segundos y reintentamos con backoff. Las consultas y los
    chunks ya vistos se sirven desde caché (útil al reconstruir índices en B2/C4).
    """

    def __init__(self, base, timeout: float = 12.0, retries: int = 6, batch_size: int = 16):
        self.base = base
        self.timeout = timeout
        self.retries = retries
        self.batch_size = batch_size
        self._pool = concurrent.futures.ThreadPoolExecutor(max_workers=8)
        self._query_cache: Dict[str, List[float]] = {}
        self._doc_cache: Dict[str, List[float]] = {}
        self.stats = {"llamadas": 0, "reintentos": 0, "cache_consultas": 0, "cache_chunks": 0}

    def _call(self, fn, arg):
        last_error = None
        for attempt in range(1, self.retries + 1):
            self.stats["llamadas"] += 1
            future = self._pool.submit(fn, arg)
            try:
                return future.result(timeout=self.timeout)
            except Exception as error:  # TimeoutError, 502, etc.
                last_error = error
                self.stats["reintentos"] += 1
                print(f"   ↻ embeddings: intento {attempt}/{self.retries} falló ({type(error).__name__})")
                time.sleep(min(2 ** attempt, 10))
        raise RuntimeError(f"Embeddings fallaron tras {self.retries} intentos") from last_error

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        missing = [t for t in dict.fromkeys(texts) if t not in self._doc_cache]
        self.stats["cache_chunks"] += len(texts) - len(missing)
        for i in range(0, len(missing), self.batch_size):
            batch = missing[i:i + self.batch_size]
            for text, vector in zip(batch, self._call(self.base.embed_documents, batch)):
                self._doc_cache[text] = vector
        return [self._doc_cache[t] for t in texts]

    def embed_query(self, text: str) -> List[float]:
        if text in self._query_cache:
            self.stats["cache_consultas"] += 1
            return self._query_cache[text]
        vector = self._call(self.base.embed_query, text)
        self._query_cache[text] = vector
        return vector


def make_embeddings() -> ResilientEmbeddings:
    return ResilientEmbeddings(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))


# Nemotron 3.5 razona por defecto y ese razonamiento gasta max_completion_tokens;
# con enable_thinking=False la respuesta llega completa en `content`.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}


def make_llm(max_completion_tokens: int = 700, thinking: bool = False) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=max_completion_tokens,
        model_kwargs={} if thinking else NO_THINKING,
    )

In [ ]:
# Verificación del problema conocido: ¿el razonamiento deja vacía la respuesta?
probe = "Responde en una frase: ¿qué es un vector de embeddings?"
for thinking in (True, False):
    t0 = time.time()
    try:
        msg = make_llm(max_completion_tokens=120, thinking=thinking).invoke(probe)
        content = (msg.content or "").strip()
        reasoning = msg.additional_kwargs.get("reasoning_content") or ""
        finish = (msg.response_metadata or {}).get("finish_reason")
        print(f"thinking={thinking!s:5} | {time.time() - t0:4.1f}s | finish={finish} | "
              f"len(content)={len(content):3} | len(reasoning)={len(reasoning):3} | {content[:90]!r}")
    except Exception as error:
        print(f"thinking={thinking}: {type(error).__name__}: {str(error)[:200]}")

---
## Parte 1 — Descargar y convertir los PDFs

Se descargan en `papers/` (ignorado por git). Si arXiv falla, se usa la copia del repositorio del curso.

In [ ]:
import requests
from markitdown import MarkItDown

DOCS_DIR = ROOT / "papers"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
COURSE_RAW = "https://raw.githubusercontent.com/EAFIT-IA/si3003-artificial-intelligence/main/notebooks/lecture10/papers/"

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(urls: List[str], destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return
    for url in urls:
        try:
            response = requests.get(url, timeout=90)
            response.raise_for_status()
            if not response.content.startswith(b"%PDF"):
                raise ValueError("no es un PDF")
            destination.write_bytes(response.content)
            print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB) desde {url}")
            return
        except Exception as error:
            print(f"   falló {url}: {error}")
    raise RuntimeError(f"No se pudo descargar {destination.name}")


for filename, url in PAPERS.items():
    download_file([url, COURSE_RAW + filename], DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    text = md_converter.convert(str(path)).text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")
PAPER_TEXT = {"attention_paper.pdf": attention_text, "rag_paper.pdf": rag_text}

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")

In [ ]:
# Calidad de la extracción: MarkItDown (pdfminer) pega palabras en muchas líneas.
for name, text in PAPER_TEXT.items():
    tokens = text.split()
    glued = [t for t in tokens if len(t) >= 25 and t.isascii() and t.replace(",", "").replace(".", "").isalpha()]
    print(f"{name}: {len(tokens):,} tokens; {len(glued):,} 'palabras' de ≥25 letras sin espacios ({len(glued)/len(tokens):.1%})."
          f" Ej.: {glued[:2]}")

---
## Parte 2 — Convertir a `Document` y aplicar chunking

Chunks de 900 caracteres con solapamiento de 120; se descartan los de menos de 30 caracteres.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

TITLES = {"attention_paper.pdf": "Attention Is All You Need", "rag_paper.pdf": "Retrieval-Augmented Generation"}


def pdf_to_documents(text: str, title: str, source: str, chunk_size: int = 900, chunk_overlap: int = 120) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


def make_all_docs(chunk_size: int = 900, chunk_overlap: int = 120) -> List[Document]:
    return [doc for source, text in PAPER_TEXT.items()
            for doc in pdf_to_documents(text, TITLES[source], source, chunk_size, chunk_overlap)]


attention_docs = pdf_to_documents(attention_text, title="Attention Is All You Need", source="attention_paper.pdf")
rag_docs = pdf_to_documents(rag_text, title="Retrieval-Augmented Generation", source="rag_paper.pdf")
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)

---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

Colección `academic_papers_nvidia`, distancia coseno, dimensión obtenida del modelo.

In [ ]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = make_embeddings()
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

t0 = time.time()
qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)
vectorstore = QdrantVectorStore(client=qdrant_client, collection_name=COLLECTION, embedding=embeddings)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant en {time.time() - t0:.1f}s")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")

---
## Parte 4 — Pipeline RAG con LCEL

Retriever con k=5 y `score_threshold=0.25`; `format_docs` antepone `[título | archivo]`; prompt del sistema en inglés.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = make_llm(max_completion_tokens=800)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


SYSTEM_PROMPT = (
    "Answer using only the supplied academic-paper excerpts. "
    "If the excerpts do not contain enough evidence, say so explicitly. "
    "Answer in the same language as the question and be concise. "
    "Mention the paper title supporting the answer."
)


def make_prompt(system_prompt: str) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
    ])


prompt = make_prompt(SYSTEM_PROMPT)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)

---
## Parte 5 — Quiz sobre los papers (B1)

`ask()` ejecuta la cadena original, imprime la respuesta y además los chunks recuperados con su score (coseno) para saber si vinieron del paper correcto. El **chequeo automático** solo mira si la respuesta contiene las palabras clave del ground truth; la calificación sí/parcial/no se hace leyendo la respuesta.

In [ ]:
QUIZ = {
    "Q1": dict(q="What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?",
               source="attention_paper.pdf", keywords=["28.4"]),
    "Q2": dict(q="How does RAG combine parametric and non-parametric memory?",
               source="rag_paper.pdf", keywords=["parametric", "non-parametric"]),
    "Q3": dict(q="What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?",
               source="attention_paper.pdf", keywords=["softmax", "gradient"]),
    "Q4": dict(q="What are the two RAG model variants proposed in the paper and how do they differ?",
               source="rag_paper.pdf", keywords=["RAG-Sequence", "RAG-Token"]),
    "Q5": dict(q="How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?",
               source="attention_paper.pdf", keywords=["8", "64"]),
}
RESULTS = {}


def keyword_check(answer: str, keywords: List[str]) -> bool:
    flat = answer.lower().replace(" ", "").replace("-", "")
    return all(k.lower().replace(" ", "").replace("-", "") in flat for k in keywords)


def retrieved_with_scores(question: str, store=None, k: int = 5):
    store = store or vectorstore
    return store.similarity_search_with_score(question, k=k)


def ask(qid: str, question: str | None = None, chain=None, keywords=None, expected_source=None, show=True):
    item = QUIZ.get(qid, {})
    question = question or item["q"]
    keywords = keywords if keywords is not None else item.get("keywords", [])
    expected_source = expected_source or item.get("source")
    t0 = time.time()
    answer = (chain or rag_chain).invoke(question)
    elapsed = time.time() - t0
    docs = retriever.invoke(question)      # lo que vio el LLM (umbral 0.25 en escala de relevancia)
    scored = retrieved_with_scores(question)
    sources = {d.metadata["source"] for d in docs}
    RESULTS[qid] = dict(question=question, answer=answer, docs=docs, scored=scored, seconds=elapsed,
                        auto=keyword_check(answer, keywords) if keywords else None,
                        right_paper=(expected_source in sources) if expected_source else None)
    if show:
        print(f"Q: {question}\nA: {answer}\n")
        print(f"Chunks entregados al LLM: {len(docs)} | {elapsed:.1f}s")
        for doc, score in scored:
            print(f"  [{doc.metadata['title']} | {doc.metadata['source']} | chunk {doc.metadata['chunk_index']}] coseno={score:.3f}")
        if keywords:
            print(f"Chequeo automático (palabras clave {keywords}): {'✅' if RESULTS[qid]['auto'] else '❌'}")
    return answer

In [ ]:
# ── Pregunta 1 ────────────────────────────────────────────────
answer1 = ask("Q1")

<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 2 ────────────────────────────────────────────────
answer2 = ask("Q2")

<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 3 ────────────────────────────────────────────────
answer3 = ask("Q3")

<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 4 ────────────────────────────────────────────────
answer4 = ask("Q4")

<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 5 ────────────────────────────────────────────────
answer5 = ask("Q5")

<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Se imprimen los títulos y archivos de los fragmentos recuperados.

In [ ]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = ask("BONUS", q_bonus, keywords=["BART"], expected_source="rag_paper.pdf", show=False)
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print("Fuentes recuperadas:")
for (title, source), (doc, score) in zip(sources, RESULTS["BONUS"]["scored"]):
    print(f"  [{title}] — {source} | chunk {doc.metadata['chunk_index']} | coseno={score:.3f}")
print("\n¿Aparecieron los dos papers?", "sí" if len({s for _, s in sources}) == 2 else "no")

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".
</details>

### B1 · Tabla resumen

In [ ]:
def short(text: str, n: int = 200) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + "…"


print("| ID | Respuesta del RAG | Chequeo automático | Papers en el contexto | ¿Paper correcto? |")
print("|---|---|---|---|---|")
for qid in ["Q1", "Q2", "Q3", "Q4", "Q5", "BONUS"]:
    r = RESULTS[qid]
    papers = sorted({d.metadata["source"] for d in r["docs"]})
    print(f"| {qid} | {short(r['answer'])} | {'✅' if r['auto'] else '❌'} | {', '.join(papers)} | {'sí' if r['right_paper'] else 'no'} |")

---
## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

- `build_store(chunk_size)` construye un índice nuevo (overlap ≈ 13 % del chunk).
- `build_chain(store, k, score_threshold)` arma la cadena y devuelve `{question, documents, answer}`.
- 9 configuraciones: chunk 400/900/1500 (k=5, umbral 0.25) · k 3/5/8 (chunk 900, umbral 0.25) · umbral 0.15/0.25/0.40 (chunk 900, k=5).
- **Aciertos (de 5)** = respuestas de Q1–Q5 que contienen todas las palabras clave del ground truth (proxy automático, mismo criterio para todas las configuraciones).

In [ ]:
import csv

STORES = {}


def build_store(chunk_size: int, overlap_ratio: float = 0.13):
    if chunk_size in STORES:
        return STORES[chunk_size]
    docs = make_all_docs(chunk_size, int(round(chunk_size * overlap_ratio)))
    client = QdrantClient(":memory:")
    name = f"papers_{chunk_size}"
    client.create_collection(name, vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    t0 = time.time()
    store.add_documents(docs)
    STORES[chunk_size] = (store, docs, time.time() - t0)
    return STORES[chunk_size]


def build_chain(store, k: int = 5, score_threshold: float = 0.25, system_prompt: str = SYSTEM_PROMPT, llm_=None):
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": score_threshold})
    answer = ({"context": lambda x: format_docs(x["documents"]), "question": lambda x: x["question"]}
              | make_prompt(system_prompt) | (llm_ or llm) | StrOutputParser())
    return RunnableParallel(question=RunnablePassthrough(), documents=retr).assign(answer=answer)


CONFIGS = ([("chunk_size", c, 5, 0.25) for c in (400, 900, 1500)]
           + [("k", 900, k, 0.25) for k in (3, 5, 8)]
           + [("score_threshold", 900, 5, t) for t in (0.15, 0.25, 0.40)])

b2_rows, cache = [], {}
for varied, chunk_size, k, thr in CONFIGS:
    key = (chunk_size, k, thr)
    if key not in cache:       # la config base (900, 5, 0.25) aparece 3 veces: se ejecuta una sola
        store, docs, build_s = build_store(chunk_size)
        chain = build_chain(store, k, thr)
        per_q = []
        for qid in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
            out = chain.invoke(QUIZ[qid]["q"])
            per_q.append(dict(qid=qid, ok=keyword_check(out["answer"], QUIZ[qid]["keywords"]),
                              n_docs=len(out["documents"]),
                              right=QUIZ[qid]["source"] in {d.metadata["source"] for d in out["documents"]},
                              answer=out["answer"]))
        cache[key] = (len(docs), build_s, per_q)
    n_chunks, build_s, per_q = cache[key]
    b2_rows.append(dict(variable=varied, chunk_size=chunk_size, overlap=int(round(chunk_size * 0.13)), k=k,
                        score_threshold=thr, chunks=n_chunks, aciertos=sum(p["ok"] for p in per_q),
                        paper_correcto=sum(p["right"] for p in per_q),
                        docs_promedio=sum(p["n_docs"] for p in per_q) / 5,
                        fallos=",".join(p["qid"] for p in per_q if not p["ok"]) or "—",
                        indexacion_s=round(build_s, 1)))

print("| Variable | chunk_size | overlap | k | threshold | chunks | aciertos (de 5) | paper correcto (de 5) | docs al LLM (prom.) | fallos |")
print("|---|---|---|---|---|---|---|---|---|---|")
for r in b2_rows:
    print(f"| {r['variable']} | {r['chunk_size']} | {r['overlap']} | {r['k']} | {r['score_threshold']} | {r['chunks']} | "
          f"{r['aciertos']} | {r['paper_correcto']} | {r['docs_promedio']:.1f} | {r['fallos']} |")

with open(EVIDENCIAS / "B2_resultados.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(b2_rows[0]))
    writer.writeheader()
    writer.writerows(b2_rows)

In [ ]:
# ¿En qué escala compara score_threshold? Relevancia de LangChain vs coseno de Qdrant.
store900 = build_store(900)[0]
q = QUIZ["Q1"]["q"]
cos = store900.similarity_search_with_score(q, k=200)
rel = store900.similarity_search_with_relevance_scores(q, k=200)
cos_vals = [s for _, s in cos]
rel_vals = [s for _, s in rel]
print(f"coseno     : max={max(cos_vals):.3f} min={min(cos_vals):.3f}")
print(f"relevancia : max={max(rel_vals):.3f} min={min(rel_vals):.3f}")
print("¿relevancia == (coseno+1)/2?", all(abs(r - (c + 1) / 2) < 1e-6 for c, r in zip(cos_vals, rel_vals)))
for thr in (0.15, 0.25, 0.40, 0.60, 0.70):
    print(f"score_threshold={thr:.2f} ⇒ deja pasar coseno ≥ {2 * thr - 1:+.2f} → "
          f"{sum(r >= thr for r in rel_vals)}/{len(rel_vals)} de los 200 chunks más cercanos")

---
## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

Tres preguntas plausibles cuya respuesta **no** está en los papers (verificado buscando en el texto extraído: no aparecen «Romanian», «TPU» ni «CNN»). Clasificación: **admite** (dice que no hay evidencia), **inventa** (da un dato que no está), **mezcla** (usa un dato real de otro experimento como si respondiera).

In [ ]:
TRAPS = [
    "What BLEU score did the Transformer big model achieve on the WMT 2016 English-to-Romanian task?",
    "How many TPU v4 chips were used to train the base Transformer model?",
    "What ROUGE-L score did RAG obtain on the CNN/DailyMail summarization benchmark?",
]
import re
for term in ["Romanian", "TPU", "CNN"]:   # palabra completa y sensible a mayúsculas ("tpu" está dentro de "output")
    pattern = re.compile(r"\b" + term + r"\b")
    print(f"'{term}' en el texto: attention={bool(pattern.search(attention_text))} rag={bool(pattern.search(rag_text))}")

REFUSAL_EN = ["do not contain", "does not contain", "don't contain", "not mentioned", "no information",
              "not provided", "not enough evidence", "insufficient", "cannot be determined", "not specified",
              "do not provide", "does not provide", "do not mention", "does not mention", "not available"]


def admits(answer: str) -> bool:
    return any(m in answer.lower() for m in REFUSAL_EN)


base_chain = build_chain(vectorstore, 5, 0.25)
b3_before = []
for trap in TRAPS:
    out = base_chain.invoke(trap)
    b3_before.append(out)
    print(f"\nQ: {trap}\nA: {out['answer']}\n→ heurística: {'admite' if admits(out['answer']) else 'REVISAR (posible inventa/mezcla)'}"
          f" | papers en contexto: {sorted({d.metadata['source'] for d in out['documents']})}")

In [ ]:
import difflib

SYSTEM_PROMPT_B3 = (
    "Answer using only the supplied academic-paper excerpts. "
    "If the excerpts do not explicitly state the requested number, dataset, hardware or fact, reply exactly: "
    "\"The provided excerpts do not contain this information.\" "
    "Never use outside knowledge and never substitute a result from a different dataset, task or experiment. "
    "Answer in the same language as the question and be concise. "
    "Mention the paper title supporting the answer."
)
print("\n".join(difflib.unified_diff(SYSTEM_PROMPT.split(". "), SYSTEM_PROMPT_B3.split(". "),
                                     "SYSTEM_PROMPT", "SYSTEM_PROMPT_B3", lineterm="")))

strict_chain = build_chain(vectorstore, 5, 0.25, system_prompt=SYSTEM_PROMPT_B3)
print("\n| Pregunta trampa | Antes | Heurística antes | Después (prompt B3) | Heurística después |")
print("|---|---|---|---|---|")
b3_after = []
for trap, before in zip(TRAPS, b3_before):
    after = strict_chain.invoke(trap)
    b3_after.append(after)
    print(f"| {trap} | {short(before['answer'], 180)} | {'admite' if admits(before['answer']) else 'revisar'} | "
          f"{short(after['answer'], 180)} | {'admite' if admits(after['answer']) else 'revisar'} |")

# Control: ¿el prompt estricto rompe las preguntas que sí tienen respuesta?
ok_strict = sum(keyword_check(strict_chain.invoke(QUIZ[q]["q"])["answer"], QUIZ[q]["keywords"]) for q in QUIZ)
print(f"\nControl Q1–Q5 con el prompt B3: {ok_strict}/5 aciertos por palabra clave")

---
## B4 · Preguntas en español sobre papers en inglés

Q1, Q3 y Q4 traducidas. Se comparan los 5 chunks más cercanos (por `source` + `chunk_index`), sus scores y la respuesta.

In [ ]:
import numpy as np

SPANISH = {
    "Q1": "¿Qué puntaje BLEU obtuvo el Transformer (modelo grande) en WMT 2014 inglés-alemán?",
    "Q3": "¿Cuál es el propósito del factor de escala sqrt(dk) en la atención de producto punto escalado?",
    "Q4": "¿Cuáles son las dos variantes del modelo RAG propuestas en el artículo y en qué se diferencian?",
}


def cosine(a, b) -> float:
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print("| ID | chunks en común (top-5) | coseno top-1 EN | coseno top-1 ES | coseno(query EN, query ES) | auto EN | auto ES |")
print("|---|---|---|---|---|---|---|")
b4 = {}
for qid, q_es in SPANISH.items():
    q_en = QUIZ[qid]["q"]
    en = retrieved_with_scores(q_en)
    es = retrieved_with_scores(q_es)
    ids_en = {(d.metadata["source"], d.metadata["chunk_index"]) for d, _ in en}
    ids_es = {(d.metadata["source"], d.metadata["chunk_index"]) for d, _ in es}
    ans_es = rag_chain.invoke(q_es)
    ans_en = RESULTS[qid]["answer"]
    b4[qid] = dict(es=q_es, answer_es=ans_es, common=len(ids_en & ids_es))
    print(f"| {qid} | {len(ids_en & ids_es)}/5 | {en[0][1]:.3f} | {es[0][1]:.3f} | "
          f"{cosine(embeddings.embed_query(q_en), embeddings.embed_query(q_es)):.3f} | "
          f"{'✅' if keyword_check(ans_en, QUIZ[qid]['keywords']) else '❌'} | {'✅' if keyword_check(ans_es, QUIZ[qid]['keywords']) else '❌'} |")

for qid, r in b4.items():
    print(f"\n{qid} ES: {r['es']}\nA: {r['answer_es']}")

In [ ]:
# Comprobación de la hipótesis: ¿la traducción queda más cerca de su original que de otra pregunta en inglés?
qs_en = {qid: embeddings.embed_query(QUIZ[qid]["q"]) for qid in SPANISH}
qs_es = {qid: embeddings.embed_query(q) for qid, q in SPANISH.items()}
print("coseno(ES_i, EN_j)   " + "  ".join(f"EN_{j}" for j in SPANISH))
for i in SPANISH:
    print(f"ES_{i}              " + "  ".join(f"{cosine(qs_es[i], qs_en[j]):.3f}" for j in SPANISH))

---
## B5 · Mi propio quiz con ground truth

Tres preguntas nuevas; el ground truth se verificó buscando las cifras en el texto extraído de los PDFs. La tercera combina los dos papers.

In [ ]:
answer_B5_1 = ask('B5-1', 'How many training steps did the big Transformer models need, how long did training take and on which hardware?', keywords=['300,000', '3.5', 'P100'], expected_source='attention_paper.pdf')

<details>
<summary>✅ Ground Truth — B5-1</summary>

Los modelos grandes se entrenaron **300,000 pasos (3.5 días)** en una máquina con **8 GPUs NVIDIA P100** (cada paso ≈ 1.0 s). *(Fuente: Section 5.2 Hardware and Schedule — Attention Is All You Need)*
</details>

In [ ]:
answer_B5_2 = ask('B5-2', "How is the Wikipedia knowledge source prepared for RAG's non-parametric memory and how many documents does the index contain?", keywords=['100', '21'], expected_source='rag_paper.pdf')

<details>
<summary>✅ Ground Truth — B5-2</summary>

Se usa el volcado de Wikipedia de **diciembre de 2018**; cada artículo se divide en **chunks disjuntos de 100 palabras**, para un total de **21M documentos**, indexados con FAISS (MIPS con aproximación HNSW). *(Fuente: Section 3 Experiments — RAG paper)*
</details>

In [ ]:
answer_B5_3 = ask('B5-3', "Which optimizer is used to train both the Transformer in 'Attention Is All You Need' and the RAG models, and which beta values does the Transformer paper report for it?", keywords=['Adam', '0.98'], expected_source=None)

<details>
<summary>✅ Ground Truth — B5-3</summary>

Ambos usan **Adam**. El Transformer usa β1 = 0.9, **β2 = 0.98** y ε = 10⁻⁹ con warmup de 4000 pasos *(Section 5.3 Optimizer — Attention)*; RAG minimiza la log-verosimilitud marginal negativa con *stochastic gradient descent with Adam* *(Section 2.4 Training — RAG paper)*.
</details>

In [ ]:
for needle, paper in [("300,000", "attention_paper.pdf"), ("P100", "attention_paper.pdf"), ("21M", "rag_paper.pdf"),
                      ("100-word", "rag_paper.pdf"), ("0.98", "attention_paper.pdf"), ("Adam", "rag_paper.pdf")]:
    print(f"'{needle}' aparece en {paper}: {needle in PAPER_TEXT[paper]}")
print()
print("| ID | Chequeo automático | Papers en el contexto |")
print("|---|---|---|")
for qid in ["B5-1", "B5-2", "B5-3"]:
    r = RESULTS[qid]
    print(f"| {qid} | {'✅' if r['auto'] else '❌'} | {', '.join(sorted({d.metadata['source'] for d in r['docs']}))} |")

---
## C3 · App Gradio para los papers con filtro por paper

Dropdown «Todos» / «Attention Is All You Need» / «Retrieval-Augmented Generation». El filtro es un `models.Filter` de Qdrant sobre `metadata.source`; las fuentes muestran título, archivo y score.

In [ ]:
import subprocess
from IPython.display import Image, display

TOMAR_CAPTURAS = os.getenv("TOMAR_CAPTURAS") == "1"


def launch(demo, port: int):
    """Lanza Gradio en localhost sin bloquear la celda (share=True solo en Colab)."""
    _, local_url, share_url = demo.queue().launch(
        server_name="127.0.0.1",
        server_port=port,
        share=IN_COLAB,
        prevent_thread_lock=True,
        show_error=True,
        quiet=True,
    )
    print("App en", share_url or local_url)
    return local_url


def capture(url: str, filename: str, questions: List[str], dropdown: str | None = None):
    """Captura la app con Playwright (scripts/capture_gradio.py) y la muestra aquí."""
    out = EVIDENCIAS / filename
    if not TOMAR_CAPTURAS:
        print("TOMAR_CAPTURAS != 1: sin captura automática (toma una manual en Colab).")
        return
    cmd = [sys.executable, str(ROOT / "scripts" / "capture_gradio.py"), url, str(out)]
    for question in questions:
        cmd += ["--question", question]
    if dropdown:
        cmd += ["--dropdown", dropdown]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    print(result.stdout.strip() or result.stderr.strip()[-800:])
    if out.exists():
        display(Image(filename=str(out), width=900))

In [ ]:
import gradio as gr

PAPER_FILTER = {"Todos": None, "Attention Is All You Need": "attention_paper.pdf",
                "Retrieval-Augmented Generation": "rag_paper.pdf"}


def source_filter(source: str | None):
    if source is None:
        return None
    return models.Filter(must=[models.FieldCondition(key="metadata.source", match=models.MatchValue(value=source))])


answer_chain = make_prompt(SYSTEM_PROMPT) | llm | StrOutputParser()


def answer_with_filter(question: str, paper: str, k: int = 5):
    scored = vectorstore.similarity_search_with_score(question, k=k, filter=source_filter(PAPER_FILTER[paper]))
    answer = answer_chain.invoke({"context": format_docs([d for d, _ in scored]), "question": question})
    return answer, scored


def chat_papers(question: str, history: list | None, paper: str):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."
    try:
        answer, scored = answer_with_filter(question, paper)
        sources_md = f"**Fuentes ({paper}):**\n" + "\n".join(
            f"- **{d.metadata['title']}** · `{d.metadata['source']}` · chunk {d.metadata['chunk_index']} · score `{s:.3f}`"
            for d, s in scored)
    except Exception as error:
        answer, sources_md = "Error consultando el pipeline.", f"**Error:** `{type(error).__name__}: {error}`"
    return "", history + [{"role": "user", "content": question}, {"role": "assistant", "content": answer}], sources_md


with gr.Blocks(title="RAG sobre papers") as demo_c3:
    gr.Markdown("# RAG sobre papers · filtro por paper\n**Qdrant + NVIDIA NIM** · filtro `metadata.source`")
    with gr.Row():
        with gr.Column(scale=1):
            paper_box = gr.Dropdown(list(PAPER_FILTER), value="Todos", label="Paper")
            gr.Markdown(f"*{len(all_docs)} chunks indexados (900/120)*\n\n**LLM:** `{GENERATION_MODEL}`  \n**Embeddings:** `{EMBEDDING_MODEL}`")
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=380, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(label="Pregunta", lines=2, scale=4)
                send_button = gr.Button("Enviar", variant="primary", scale=1)
            sources_box = gr.Markdown()
    gr.Examples([["Which optimizer was used for training?"], [QUIZ["Q4"]["q"]]], inputs=question_box)
    send_button.click(chat_papers, [question_box, chatbot, paper_box], [question_box, chatbot, sources_box])
    question_box.submit(chat_papers, [question_box, chatbot, paper_box], [question_box, chatbot, sources_box])

url_c3 = launch(demo_c3, 7864)
capture(url_c3, "C3_gradio.png", ["Which optimizer was used for training?"], dropdown="Retrieval-Augmented Generation")

In [ ]:
# Comparación con y sin filtro
C3_QUESTIONS = ["Which optimizer was used for training?", "What is the role of beam search in decoding?"]
for question in C3_QUESTIONS:
    print(f"\n### {question}")
    for paper in PAPER_FILTER:
        answer, scored = answer_with_filter(question, paper)
        srcs = [f"{d.metadata['source'].split('_')[0]}#{d.metadata['chunk_index']}({s:.2f})" for d, s in scored]
        print(f"- {paper:32} | fuentes: {', '.join(srcs)}\n  respuesta: {short(answer, 260)}")

---
## C5 · MMR frente a similitud pura

`search_type="mmr"` con k=5, fetch_k=20 y lambda_mult=0.5. **Redundancia** = coseno medio entre pares de los 5 chunks devueltos (más alto = más repetidos). **Cobertura** = papers distintos y si algún chunk contiene las palabras clave del ground truth.

In [ ]:
from itertools import combinations

sim_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})

C5_QUESTIONS = [
    ("Q4", QUIZ["Q4"]["q"], QUIZ["Q4"]["keywords"]),
    ("BONUS", q_bonus, ["BART"]),
    ("Q3", QUIZ["Q3"]["q"], QUIZ["Q3"]["keywords"]),
]


def redundancy(docs) -> float:
    vecs = embeddings.embed_documents([d.page_content for d in docs])
    pairs = list(combinations(range(len(vecs)), 2))
    return sum(cosine(vecs[i], vecs[j]) for i, j in pairs) / len(pairs)


def keyword_in_docs(docs, keywords) -> bool:
    text = " ".join(d.page_content for d in docs).lower().replace(" ", "")
    return all(k.lower().replace(" ", "") in text for k in keywords)


print("| Pregunta | Método | chunks (paper#idx) | redundancia | papers | ¿palabras clave en contexto? | respuesta ok |")
print("|---|---|---|---|---|---|---|")
c5 = {}
for qid, question, kws in C5_QUESTIONS:
    for name, retr in (("similitud", sim_retriever), ("MMR", mmr_retriever)):
        docs = retr.invoke(question)
        answer = (make_prompt(SYSTEM_PROMPT) | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        c5[(qid, name)] = dict(docs=docs, answer=answer)
        ids = ", ".join(f"{d.metadata['source'][:3]}#{d.metadata['chunk_index']}" for d in docs)
        print(f"| {qid} | {name} | {ids} | {redundancy(docs):.3f} | {len({d.metadata['source'] for d in docs})} | "
              f"{'sí' if keyword_in_docs(docs, kws) else 'no'} | {'✅' if keyword_check(answer, kws) else '❌'} |")

In [ ]:
print("Estadísticas de ResilientEmbeddings:", embeddings.stats)
demo_c3.close()

---
## Reflexión final (R1)

Las respuestas, apoyadas en las salidas de este notebook, están en `RESPUESTAS.md` (sección R1):

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?**
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tuvo modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?**